In [ ]:
##Question 1
import pandas as pd
ROLL_NUMBER = "1024170205"
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

custom_entries = [
    # digit 0 -> billing
    {"question": "why was my late payment fee charged",
     "answer": "A late fee is added if payment is made after the due date shown on your invoice.",
     "keywords": "late fee penalty", "category": "billing"},
    # digit 5 -> general
    {"question": "what are the weekend branch timings",
     "answer": "Branches are open 10 AM to 2 PM on Saturdays and closed on Sundays.",
     "keywords": "weekend saturday sunday", "category": "general"},
]

df = pd.DataFrame(fixed_entries + custom_entries)
print("Final 6 row DataFrame")
print(df.to_string())

Final 6 row DataFrame
                              question                                                                            answer                 keywords category
0               what is the annual fee                                                         The annual fee is Rs 500.    fee cost price charge  billing
1                how to reset password                                                  Go to Settings > Reset Password.     password reset login  account
2          what are your working hours                                                         We are open 9 AM to 5 PM.   hours timing open time  general
3                how can i pay the fee                                        You can pay via UPI, card, or net banking.      pay payment upi fee  billing
4  why was my late payment fee charged  A late fee is added if payment is made after the due date shown on your invoice.         late fee penalty  billing
5  what are the weekend branch timings          

In [ ]:
##Question 2
def score_query(query, df):
    words = query.lower().split()
    results = []
    for i in range(len(df)):
        keywords = df.loc[i, "keywords"].split()
        matched = [w for w in words if w in keywords]
        if matched:
            results.append({
                "question": df.loc[i, "question"],
                "answer": df.loc[i, "answer"],
                "score": len(matched),
                "confidence": round(len(matched) / len(words), 2),
            })
    results.sort(key=lambda r: r["score"], reverse=True)
    return pd.DataFrame(results)


print("\nQuery 'reset my password'")
print(score_query("reset my password", df).to_string())


Q2: Query 'reset my password'
                question                            answer  score  confidence
0  how to reset password  Go to Settings > Reset Password.      2        0.67


In [ ]:
##Question 3
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"].tolist()


print("\nQuestions in category 'billing'")
print(same_category("billing", df))


Questions in category 'billing'
['what is the annual fee', 'how can i pay the fee', 'why was my late payment fee charged']


In [ ]:
##Question 4
entry_index = 0
new_keyword = input("\nEnter a new keyword for '" + df.loc[entry_index, "question"] + "': ").strip().lower()

keywords = df.loc[entry_index, "keywords"].split()
if new_keyword and new_keyword not in keywords:
    keywords.append(new_keyword)
df.loc[entry_index, "keywords"] = " ".join(keywords)

df.to_csv(ROLL_NUMBER + "_faq_data.csv", index=False)
print("Updated keywords:", df.loc[entry_index, "keywords"])
print("Saved to", ROLL_NUMBER + "_faq_data.csv")


Enter a new keyword for 'what is the annual fee': Yearly Amt
Updated keywords: fee cost price charge amt yearly amt
Saved to 1024170205_faq_data.csv


In [ ]:
##Question 5
print(df.groupby("category").size())

category
account    1
billing    3
general    2
dtype: int64


In [ ]:
##Question 6
def score_with_ties(query, df):
    results = score_query(query, df)
    if results.empty:
        print("No matching entries found.")
        return
    top = results["score"].max()
    best = results[results["score"] == top]
    if len(best) > 1:
        print("TIE: " + str(len(best)) + " entries share the top score of " + str(top) + ":")
    else:
        print("Single best match:")
    print(best.to_string())


print("\nQuery with a tie ('fee')")
score_with_ties("fee", df)

print("\nQuery without a tie ('reset password login')")
score_with_ties("reset password login", df)


Query with a tie ('fee')
TIE: 3 entries share the top score of 1:
                              question                                                                            answer  score  confidence
0               what is the annual fee                                                         The annual fee is Rs 500.      1         1.0
1                how can i pay the fee                                        You can pay via UPI, card, or net banking.      1         1.0
2  why was my late payment fee charged  A late fee is added if payment is made after the due date shown on your invoice.      1         1.0

Query without a tie ('reset password login')
Single best match:
                question                            answer  score  confidence
0  how to reset password  Go to Settings > Reset Password.      3         1.0
